#Cohere CHAT Endpoint

In this notebook, you'll build a chatbot using Cohere's Chat endpoint. By the end of this notebook, you'll be able to build a simple chatbot that can respond to user messages and maintain the context of the conversation.

*Read the accompanying [blog post here](https://docs.cohere.com/docs/building-a-chatbot).

##Overview

We'll do the following steps:



- **Step 1: Quickstart** - Learn the quickest and easiest way to call the Chat endpoint.
- **Step 2: Adding a preamble** - Steer a chatbot's response toward certain styles, personas, or other characteristics.
- **Step 3: Streaming the responses** - Display a chatbot's response incrementally as it is generated, as opposed to waiting for the entire
response to be completed.
- **Step4: Building the Chat History** - Explore two different options for getting the chatbot to maintain the context of the conversation.
markdown

##Setup

We'll start by installing the tools we'll need and then importing them.

In [ ]:
!pip install cohere -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 39.6 MB/s eta 0:00:00


In [4]:
import uuid
import cohere
import json
from cohere import ChatMessage

In [ ]:
co = cohere.Client("API_KEY")

##Step 1: Quickstart

The `message` parameter is the only required parameter for the Chat endpoint.

In our case, we also use the `model` parameter to specify the underlying chat model we want to use. For this, we pick Command R+, Cohere's newest large language model.

We call the endpoint with `co.chat()`, and get the main content of the response from the `text` value.

In [7]:
response = co.chat(message="Hello", model="command-a-03-2025")
print(response.text)

Hello! How can I assist you today?


## Step 2: Adding a preamble

A conversation starts with a system message, or a preamble, to help steer a chatbot's response toward certain characteristics. In the quickstart example, we didn't have to define a preamble because a default one was used.

We can define our own preamble with the `preamble` parameter.

In [8]:
response = co.chat(message="Hello",
                   model="command-a-03-2025",
                   preamble="You are an expert in artificial intelligence and generative AI, Don't use any greetings")
print(response.text)

I'm here to assist you with any questions or tasks related to artificial intelligence and generative AI. How can I help you today?


#Step 3: Streaming the response

Our examples so far generate responses in a non-streamed manner. This means that the endpoint would return a response object only after the model has generated the text in full.

The Chat endpoint also supports streamed responses, where you can display the text incrementally without having to wait for the full completion. To activate it, use `co.chat_stream()` instead of `co.chat()`.

In streaming mode, the endpoint will generate a series of objects. To get the text content, we take objects whose `event_type` is `text-generation`.

In [9]:
stream = co.chat_stream(message="Hello. I'd like to learn about techniques for text embedding",
                        model="command-a-03-2025",
                        preamble="You are an expert in artificial intelligence and generative AΙ.")

for event in stream:
  if event.event_type == "text-generation":
    print(event.text, end='')

Text embedding is a fundamental technique in natural language processing (NLP) that converts text into numerical vectors, allowing machines to understand and process human language. These embeddings capture semantic meanings, relationships, and contexts of words or sentences. Here’s an overview of key techniques for text embedding:

---

### **1. Traditional Word Embeddings**
These methods represent individual words as dense vectors in a continuous vector space.

#### **a. Word2Vec**
- **Approach**: Trains on large text corpora using predictive models (Continuous Bag of Words - CBOW or Skip-Gram).
- **Features**: Captures word semantics and relationships (e.g., "king - man + woman = queen").
- **Limitations**: Does not handle out-of-vocabulary words or polysemy (words with multiple meanings).

#### **b. GloVe (Global Vectors for Word Representation)**
- **Approach**: Combines global word co-occurrence statistics with local context windows.
- **Features**: Efficient and scalable, captur

## Step 4: Building the Chat history

At the core of a conversation is a multi-turn dialog between the user and the chatbot. This requires the chatbot to have a "memory" of all the previous turns to maintain the state of the conversation.

### Option 1: Using the conversation history persistence feature

The Chat endpoint supports state management by persisting the conversation history. As a conversation progresses, the endpoint continuously updates the conversation history. This means developers don't have to deal with the complexity and inconvenience of managing conversation history in their application.

To use this feature, use the `conversation_id` parameter, which is a unique string you assign to a conversation. We'll use the `uuid` library to do this.

Putting everything together, let's now build a simple chat interface that takes in a user message, generates the chatbot response, automatically updates the conversation history, and repeats these steps until the user quits the conversation.

As described before, in streaming mode, the Chat endpoint generates a series of objects. To get the conversation history, we take the object with `event_type` of `stream-end` and save it as a new variable `chat_history`.

In [15]:
# Create a conversation ID
conversation_id = str(uuid.uuid4())

# Define the preamle
preamble = "You are an expert in artificial intelligence and generative AI, Don't use any greetings"

print('Starting the chat. Type "quit" to end.\n')

while True:
  # User message
  message = input("User: ")

  # Typing "quit" ends the coversation
  if message.lower() == 'quit':
    print("Ending chat.")
    break

  # Chatbot response
  stream = co.chat_stream(message=message,
                          model="command-a-03-2025",
                          preamble=preamble,
                          conversation_id=conversation_id)
  print("Chatbot: ", end='')

  for event in stream:
    if event.event_type == "text-generation":
      print(event.text, end='')
    if event.event_type == "stream-end":
      chat_history = event.response.chat_history

  print(f"\n{'-'*100}\n")

Starting the chat. Type "quit" to end.



KeyboardInterrupt: Interrupted by user

In [ ]:
for chat in chat_history:
  print(chat)

## Option 2: Managing the conversation history yourself

If you opt not to use the endpoint's conversation history persistence feature, you can use the `chat_history` parameter to manage the conversation history yourself.

In [17]:
#Initialize the chat history
chat_history = []

#Define the preamble
preamble = "You are an expert in artificial intelligence and generative AI."

print('Starting the chat. Type "quit" to end.\n')

while True:

  #User message
  message = input("User: ")

  # Typing "quit" ends the conversation.
  if message.lower()== 'quit':
    print("Ending chat.")
    break

  # Chatbot response
  stream = co.chat_stream(message=message,
                          model="command-a-03-2025",
                          preamble=preamble,
                          chat_history=chat_history)
  chatbot_response = ""
  print("Chatbot: ", end='')

  for event in stream:
    if event.event_type == "text-generation":
      print(event.text, end='')
      chatbot_response += event.text
  print("\n")

  # Add to chat history
  chat_history.extend(
      [ChatMessage(role="USER", message=message),
       ChatMessage(role="CHATBOT", message=chatbot_response)]
  )

Starting the chat. Type "quit" to end.

Chatbot: Maintaining eye contact while speaking is an important aspect of effective communication and can help you appear more confident and engaging. Here are some tips to help you improve your eye contact:

1. **Practice with a Friend or in Front of a Mirror:** Start by practicing with someone you're comfortable with or by yourself in front of a mirror. Focus on looking at the person's eyes or your own reflection while speaking. This will help you get used to the feeling of maintaining eye contact.

2. **Use the 50/70 Rule:** A common guideline is the 50/70 rule, which suggests that speakers should make eye contact with their audience for 50% of the time they are speaking and 70% of the time they are listening. This ensures a balance between connecting with your audience and allowing for natural breaks in eye contact.

3. **Scan the Room:** If you're speaking to a group, try to make eye contact with different individuals for a few seconds each.

KeyboardInterrupt: 

In [18]:
for chat in chat_history:
  print(chat)

message='User: How can I maintainneye contact while speaking' tool_calls=None role='USER'
message="Maintaining eye contact while speaking is an important aspect of effective communication and can help you appear more confident and engaging. Here are some tips to help you improve your eye contact:\n\n1. **Practice with a Friend or in Front of a Mirror:** Start by practicing with someone you're comfortable with or by yourself in front of a mirror. Focus on looking at the person's eyes or your own reflection while speaking. This will help you get used to the feeling of maintaining eye contact.\n\n2. **Use the 50/70 Rule:** A common guideline is the 50/70 rule, which suggests that speakers should make eye contact with their audience for 50% of the time they are speaking and 70% of the time they are listening. This ensures a balance between connecting with your audience and allowing for natural breaks in eye contact.\n\n3. **Scan the Room:** If you're speaking to a group, try to make eye co